## Oppgave 4
### a)

### Importerer biblioteker

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import random
import plotly.graph_objects as go

In [ ]:
"""Må inkludere disse bibliotekene for å plotte."""
# !pip install pandas
# !pip install plotly

In [ ]:
"""Oppretter først funksjonene. Definerer så variablene."""

# Bruker liste til å lagre posisjonene, for da vil vi kunne bruke det som indekser.
def MakeParticles(params):   #Diverse sjekker, om posisjoner er tillatt, slik at koden fungerer.
    size = params["b"]
    count = params["Np"]
    L = params["L"]
    initParticlePos = np.linspace(0, L-2*size, count)
    listPosition = []

    for n in range(len(initParticlePos)):
        listPosition.append(int(initParticlePos[n]))
    return listPosition

def occupied(particlePos, params):
    particleCount = params["Np"]
    particleSize = params["b"]
    L = params["L"]
    occupiedMatrix = np.zeros((particleCount, 3))    
    """occupiedMatrix[i][0] = partikkels senter, occupiedMatrix[i][1] = venstre, occupiedMatrix[i][2] = høyre
    Kan endres på dersom noe annet gir mer mening."""
    for pos in range(len(particlePos)):
        occupiedMatrix[pos][0] = particlePos[pos]
        occupiedMatrix[pos][1] = (particlePos[pos] - (particleSize) // 2) % L
        occupiedMatrix[pos][2] = (particlePos[pos] + (particleSize) // 2) % L
    return occupiedMatrix



In [ ]:
#Potensialer
def V1(x, params):                #uten k fordi den inngår i p_pos
    Nx = params["Nx"]
    alfa = params["alfa"]
    periode = x % Nx

    if periode <= alfa*Nx:
        return periode/(alfa*Nx)
    else:
        return -(periode-Nx)/((1-alfa)*Nx)

def V2(x, params):                          #også uten k for samme grunn
    return 0

In [ ]:
#Henholdsvise sannsynlighetsfunksjoner p+, p0 og p-
def p_pos(x, V, params):
    beta_k = params["beta_k"]
    L = params["L"]

    #får litt numerisk overflow hvis jeg ikke clipper
    arg1 = np.clip(-beta_k*(V((x-1) % L, params) - V((x+1) % L, params)), -700, 700)
    arg2 = np.clip(-beta_k*(V(x % L, params) - V((x+1) % L, params)), -700, 700)

    nevner = 1  + np.exp(arg1) \
                + np.exp(arg2)

    return 1/nevner


def p_0(x, V, params):
    beta_k = params["beta_k"]
    L = params["L"]

    arg1 = np.clip(-beta_k*(V((x-1) % L, params) - V(x % L, params)), -700, 700)
    arg2 = np.clip(-beta_k*(V((x+1) % L, params) - V(x % L, params)), -700, 700)
    nevner = 1  + np.exp(arg1) \
                + np.exp(arg2)
    
    return 1/nevner


def p_neg(x, V, params):
    beta_k = params["beta_k"]
    L = params["L"]
    arg1 = np.clip(-beta_k*(V((x+1) % L, params) - V( (x-1) % L, params)), -700, 700)
    arg2 = np.clip(-beta_k*(V( x % L, params) - V((x-1) % L, params)), -700, 700)
    nevner = 1  + np.exp(arg1) \
                + np.exp(arg2)
    return 1/nevner

def lookUpTabell_V1(params):
    L = params["L"]
    tabell_V1_p_pos = np.zeros(L)
    tabell_V1_p_neg = np.zeros(L)

    V = V1    # Her bruker vi potensial V1!
    for x in range(L):
        tabell_V1_p_pos[x] = p_pos(x, V, params)
        tabell_V1_p_neg[x] = p_neg(x, V, params)

    return tabell_V1_p_pos, tabell_V1_p_neg

def lookUpTabell_V2(params):
    L = params["L"]
    p_pos_val = p_pos(1, V2, params)
    p_neg_val = p_neg(1, V2, params)

    tabell_V2_p_pos = np.full(L, p_pos_val)
    tabell_V2_p_neg = np.full(L, p_neg_val)

    return tabell_V2_p_pos, tabell_V2_p_neg

In [ ]:
params4a = {
    "alfa"      : 0.2,
    "Tp"        : 40,       # Tid for 0.5 periode
    "Nx"        : 20,
    "beta_k"    : 1000,
    "Np"        : 8,        # Velg et fint antall
    "b"         : 6,        # Størrelse. Velg en fin størrelse. Kan lage en funksjon her.
}
params4a["L"] = 4*params4a["Nx"]
params4a["timeSteps"] = 10*params4a["Tp"]

params4a["initPosition"] = MakeParticles(params4a)

params4a["tabell_V1_p_pos"], params4a["tabell_V1_p_neg"] = lookUpTabell_V1(params4a)
params4a["tabell_V2_p_pos"], params4a["tabell_V2_p_neg"] = lookUpTabell_V2(params4a)

In [ ]:
""" Du trenger ikke ha med dette Johannes"""

# Prøver å forstå alt sammen. Tegner potensialet.
# Må modifisere litt. Kan ikke sammenligne et helt array med periode, siden hver verdi for forskjellig bool-verdi.

alfa = 0.2
def V1_test(x):
    potensialer = np.zeros_like(x, dtype = float)    # Viktig å endre datatype. Hvis ikke blir verdiene til int.
    for i in range(len(x)):
        Hvor_i_periode = x[i] % params4a["Nx"]
        if Hvor_i_periode <= alfa*params4a["Nx"]:
            potensialer[i] = 10*Hvor_i_periode / (alfa* params4a["Nx"])
        else:
            potensialer[i] = -10*(Hvor_i_periode - params4a["Nx"])/((1-alfa)*params4a["Nx"]) 
    return potensialer
    
x_axis = np.arange(0, params4a["L"], 1)
y_potential = V1_test(x_axis)
plt.plot(x_axis, V1_test(x_axis))
plt.xlabel("x")
plt.ylabel("y")
plt.axvline(x=4, color='red', linestyle='--', linewidth=1, label='x=2')
plt.axvline(x=20, color='red', linestyle='--', linewidth=1, label='x=20')
plt.show()

In [ ]:
# Optimalisering. Siden partiklene bare kan bevege steg med lengde 1,
# så behøver vi bare å sjekke forrige/neste partikkel.

def enSimulasjon_4a(params):

    particlePos = params["initPosition"].copy()
    particleCount = params["Np"]
    timeSteps = params["timeSteps"]
    Tp = params["Tp"]
    L = params["L"]
    tabell_V1_p_pos, tabell_V1_p_neg = params4a["tabell_V1_p_pos"], params4a["tabell_V1_p_neg"]
    tabell_V2_p_pos, tabell_V2_p_neg = params4a["tabell_V2_p_pos"], params4a["tabell_V2_p_neg"]

    rightIndex = 2
    leftIndex = 1

    positionsOverTime = []
    queue = list(range(particleCount))

    
    occupiedMatrix = occupied(particlePos, params)
    positionsOverTime.append(particlePos.copy())

    for k in range(timeSteps):
        random.shuffle(queue)
        whichPotential = ( k // Tp) % 2    
        if whichPotential == 0:
            tabell_p_pos, tabell_p_neg = tabell_V1_p_pos, tabell_V1_p_neg
        else:
            tabell_p_pos, tabell_p_neg = tabell_V2_p_pos, tabell_V2_p_neg
        for index in queue:
            tilfeldigtall = np.random.rand()
            x = particlePos[index]                            # Må sjekke kollisjon for hver partikkel
            if tilfeldigtall <= tabell_p_neg[x % L]:     # Mulig kollisjon mot høyre
                if (x-1) == occupiedMatrix[index-1][rightIndex]:
                    continue
                x -= 1
            elif tilfeldigtall >= (1 - tabell_p_pos[x % L]):
                if (x+1) == occupiedMatrix[(index + 1) % particleCount][leftIndex]:
                    continue
                x += 1
            particlePos[index] = x % (L)                      # Hopper tilbake til start.
            occupiedMatrix = occupied(particlePos, params)
    
            # print(particlePos)
        positionsOverTime.append(particlePos.copy())
    return positionsOverTime

In [ ]:
positions_over_time = enSimulasjon_4a(params4a)
# print(positions_over_time)

num_balls = len(positions_over_time[0])
Tp = params4a["Tp"]
L = params4a["L"]


# Create frames
frames = []
for t, positions in enumerate(positions_over_time):
    whichPotential = ( t // Tp) % 2
    if whichPotential == 0:
        potensial = "Sagtann"
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(
                        x=positions,
                        y=[y_potential[coord] for coord in positions],
                        # mode="markers",
                        # marker=dict(size=12, symbol = "star"),
                        mode = "text",
                        text= "🚀",
                        textfont=dict(size=20),
                        name = "Random walk paritcles"
                    )
                ],
                layout= go.Layout(
                    annotations=[
                        dict(
                            text=f"Frame: {t}, potensial = {potensial}",
                            x=0.95,
                            y=0.95,
                            xref = "paper",
                            yref = "paper",
                            showarrow = False,
                            font=dict(size=20)
                        )
                    ]
                ),
                name=str(t)
            )
        )
    
    else:
        potensial = "flatt"
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(
                        x=positions,
                        y=[0]*num_balls,
                        # mode="markers", 
                        # marker=dict(size=12, symbol = "square")

                    )
                ],
                layout= go.Layout(
                    annotations=[
                        dict(
                            text=f"Frame: {t}, potensial = {potensial}",
                            x=0.95,
                            y=0.95,
                            xref = "paper",
                            yref = "paper",
                            showarrow = False,
                            font=dict(size=20)
                        )
                    ]
                ), 
                name=str(t)
            )
    )



# Initial figure
fig = go.Figure(
    data=[
        go.Scatter(
            x=positions_over_time[0],
            y=[0]* num_balls,
            mode="markers",
            marker=dict(size=12)
        )
    ],
    frames=frames
)


fig.update_layout(
    title = f'Antall partikler = {num_balls}, størrelse = {params4a["b"]}.'
)

# Add slider (frame counter)
fig.update_layout(
    sliders=[{
        "currentvalue": {"prefix": "Frame: "},
        "steps": [
            {
                "args": [[f.name], {"frame": {"duration": 50, "redraw": True},
                                    "mode": "immediate"}],
                "label": f.name,
                "method": "animate"
            }
            for f in frames
        ]
    }]
)

# Add play button
fig.update_layout(
    xaxis=dict(range=[0, L], title="Position"),
    yaxis=dict(range=[-10, 20]),
    updatemenus=[
        dict(
            type="buttons",
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[None, {"frame": {"duration": 200, "redraw": True},
                                 "fromcurrent": True}]
                )
            ]
        )
    ]
)

# Potential curve
fig.add_trace(
    go.Scatter(
        x=x_axis,
        y= y_potential if whichPotential == 0 else 0,
        mode="lines",
        name="Potential"
    )
)

fig.show()

#### Oppgave 4b)

Legger inn flere funksjoner

In [ ]:
def rho(b, Np, Ns, Nx):
    return b*Np/(Ns*Nx)


In [ ]:
params4b = {
    "beta_k"  : 1000,
    "Nx"      : 100,
    "Tp"      : 300,
    "Ns"      : 10,
    "alfa"    : 0.2,
    "b"       : 20,   # Størrelse på partiklene.
    "Nc"      : 100,  # Antall sykler. For hver tetthet.
    "Np"      : 7 # Denne skal varieres for å danne forskjellige tetthet rho.
}
params4b["rho"] = rho(params4b["b"], params4b["Np"],params4b["Ns"],params4b["Nc"])
params4b["initPositions"] = MakeParticles(params4b)

In [ ]:
def particleCurrent_sim(partiklerPos, params, tabeller):
    timeSteps = params["timeSteps"]
    Tp = params["Tp"]
    L = params["L"]
    Np = params["Np"]
    startPot_pos_table = tabeller["startPot_pos_table"]
    startPot_neg_table = tabeller["startPot_neg_table"]
    secondPot_pos_table = tabeller["secondPot_pos_table"]
    secondPot_neg_table = tabeller["secondPot_neg_table"]

    particleCurrent = np.zeros(timeSteps)

    for k in range(timeSteps):
        hvilkenPotensial = ( k // Tp) % 2    
        n_pos = 0
        n_neg = 0
        if hvilkenPotensial == 0:
            tabell_p_pos, tabell_p_neg = startPot_pos_table, startPot_neg_table
        else:
            tabell_p_pos, tabell_p_neg = secondPot_pos_table, secondPot_neg_table
        
        randomtabell = np.random.rand(Np)

        posProb = tabell_p_pos[partiklerPos]
        negProb = tabell_p_neg[partiklerPos]

        neg_flytt = (randomtabell <= negProb)
        pos_flytt = (~neg_flytt) & (randomtabell >= (1 - posProb))

        partiklerPos[neg_flytt] -= 1
        partiklerPos[pos_flytt] += 1
        partiklerPos %= L

        # for i in range(len(partiklerPos)):
        #         x = partiklerPos[i]

        #         if randomtabell[i] <= tabell_p_neg[x]:      
        #             x -= 1                                
        #             n_neg += 1                            
        #         elif randomtabell[i] >= (1 - tabell_p_pos[x]):
        #             x += 1
        #             n_pos += 1
        #         partiklerPos[i] = x % (L)

        n_pos = pos_flytt.sum()
        n_neg = neg_flytt.sum()
        particleCurrent[k] = (n_pos - n_neg)/Np

    return particleCurrent

In [ ]:
# Optimalisering. Siden partiklene bare kan bevege steg med lengde 1,
# så behøver vi bare å sjekke forrige/neste partikkel.

def enSimulasjon_4b():
    positionsOverTime = []
    particlePos = 
    rightIndex = 2
    leftIndex = 1
    n_pos = 0
    n_neg = 0
    queue = list(range(particleCount))
    particleCurrent = np.zeros(300*100)
    
    occupiedMatrix = occupied(particlePos)
    positionsOverTime.append(particlePos.copy())

    for k in range(timeSteps):
        random.shuffle(queue)
        whichPotential = ( k // Tp) % 2    
        if whichPotential == 0:
            tabell_p_pos, tabell_p_neg = tabell_V1_p_pos, tabell_V1_p_neg
        else:
            tabell_p_pos, tabell_p_neg = tabell_V2_p_pos, tabell_V2_p_neg
        for index in queue:
            tilfeldigtall = np.random.rand()
            x = particlePos[index]                            # Må sjekke kollisjon for hver partikkel
            if tilfeldigtall <= tabell_p_neg[x % L]:     # Mulig kollisjon mot høyre
                if (x-1) == occupiedMatrix[index-1][rightIndex]:
                    continue
                x -= 1
                n_neg += 1
            elif tilfeldigtall >= (1 - tabell_p_pos[x% L]):
                if (x+1) == occupiedMatrix[(index + 1) % particleCount][leftIndex]:
                    continue
                x += 1
                n_pos += 1
            particlePos[index] = x % (L)                      # Hopper tilbake til start.
            occupiedMatrix = occupied(particlePos)
        particleCurrent[k] = (n_pos - n_neg)/Np

    return particleCurrent

strøm = enSimulasjon_4b()
print(strøm)